In [1]:
import numpy as np
import pandas as pd
import os
import calendar
import datetime
import re
from math import fabs
import math
import json
from collections import defaultdict

from py_vollib.black_scholes.implied_volatility import implied_volatility
#from py_vollib.black_scholes import Call, Put


path = '/media/sirshendu/One Touch/1_SMData/2_CurrentData/1_5s_candle'

def is_valid_date(s):
    try:
        datetime.datetime.strptime(s, "%Y-%m-%d")
        return True
    except ValueError:
        return False

def last_thursday(month_abbr, year=None):
    if year is None:
        year = datetime.datetime.today().year
    # Convert month abbreviation to month number
    month_num = datetime.datetime.strptime(month_abbr, "%b").month
    
    # Get the last day of the month
    last_day = calendar.monthrange(year, month_num)[1]
    last_date = datetime.datetime(year, month_num, last_day)
    
    # Step backwards until we hit Thursday (weekday() == 3)
    while last_date.weekday() != 3:
        last_date -= datetime.timedelta(days=1)
    
    return last_date



dates = [d for d in os.listdir(path) if is_valid_date(d)]
dates = sorted(dates, key=lambda x: datetime.datetime.strptime(x, "%Y-%m-%d"))



class BelowIntrinsicException(Exception):
    pass

def getAtmCePeIV(date, stock, path=path):
    allContracts = os.listdir(path+'/'+date+'/Fno/Stock/'+stock+'/Opt')
    if len(allContracts) == 0:
        print('There is no opt contract')
        try:
            spot = pd.read_parquet(path+'/'+date+'/Spot/Stock/'+stock+'.parquet')
        except:
            try:
                spot = pd.read_csv(path+'/'+date+'/Spot/Stock/'+stock+'/'+stock+'.csv')
            except:
                spot = pd.read_parquet(path+'/'+date+'/Spot/Stock/'+stock+'/'+stock+'.parquet')
        SpotLtp = spot.set_index('Timestamp').sort_index().iloc[-1]['Close']
        return SpotLtp, 0, 0, 0, 0
    
    #pattern = re.compile(r'NSE_(\w+?)(\d{2})([A-Z]{3})(\d+)([CP]E)')
    pattern = re.compile(
        r'NSE_([A-Z0-9\-\&]+?)(\d{2})([A-Z]{3})(\d+(?:\.\d+)?)([CP]E)\.[a-zA-Z0-9]+$'
    )

    expiry_month = []
    strikes = []
    for contract in allContracts:
        #print(contract)
        m = pattern.search(contract)
        stock, day, mon, strike, opt_type = m.groups()
        
        strike = float(strike)
        if strike.is_integer():
            strike = int(strike)
        
        
        if mon not in expiry_month:
            expiry_month.append(mon)
        if strike not in strikes:
            strikes.append(strike)

    expiry = [ last_thursday(ex, 2025) for ex in expiry_month]
    #latestExpiry = sorted(expiry)[0]

    sorted_pairs = sorted(zip(expiry, expiry_month), key=lambda x: x[0], reverse=False)
    expiry, expiry_month = zip(*sorted_pairs)
    expiry = list(expiry)
    expiry_month = list(expiry_month)

    expiryDate = expiry[0]
    expiryMonth = expiry_month[0]

    try:
        spot = pd.read_parquet(path+'/'+date+'/Spot/Stock/'+stock+'.parquet')
    except:
        try:
            spot = pd.read_csv(path+'/'+date+'/Spot/Stock/'+stock+'/'+stock+'.csv')
        except:
            spot = pd.read_parquet(path+'/'+date+'/Spot/Stock/'+stock+'/'+stock+'.parquet')
    SpotLtp = spot.set_index('Timestamp').sort_index().iloc[-1]['Close']
    ATMStrike = min(strikes, key = lambda x : abs(x-SpotLtp))
    t = expiryDate-pd.to_datetime(date)
    t = t.days / 365 ## time to expiry in years


    ### then get the price of ATM CE & PE
    CeLtp = None; PeLtp = None
    for contract in allContracts:
        m = pattern.search(contract)
        stock, day, mon, strike, opt_type = m.groups()
        strike = float(strike)
        if strike.is_integer():
            strike = int(strike)
        
        if contract[-4:] == '.csv':
            CePe = contract.split('.csv')[0][-2:]
        else:
            CePe = contract.split('.parquet')[0][-2:]
        if mon == expiryMonth and strike == ATMStrike:
            if CePe == 'CE':
                if contract[-4:] == '.csv':
                    CeDf = pd.read_csv(path+'/'+date+'/Fno/Stock/'+stock+'/Opt/'+contract)
                else:
                    CeDf = pd.read_parquet(path+'/'+date+'/Fno/Stock/'+stock+'/Opt/'+contract)
                CeLtp = CeDf.set_index('Timestamp').sort_index().iloc[-1]['Close']
            elif CePe == 'PE':
                if contract[-4:] == '.csv':
                    PeDf = pd.read_csv(path+'/'+date+'/Fno/Stock/'+stock+'/Opt/'+contract)
                else:
                    PeDf = pd.read_parquet(path+'/'+date+'/Fno/Stock/'+stock+'/Opt/'+contract)
                PeLtp = PeDf.set_index('Timestamp').sort_index().iloc[-1]['Close']
            else:
                print(contract)
                print('Error : ', CePe)
        else:
            pass
            #print(date, stock, ATMStrike, expiryDate, expiryMonth)

    if CeLtp is None:
        CeLtp = 0
        AtmCeIv = 0
        print('No CE ATm contract : ', date, stock)
    elif CeLtp < fabs(max(SpotLtp - ATMStrike, 0.0)):
        AtmCeIv = 0
        print(date, stock, ' Celtp < intrinsic')
    else:
        #print(CeLtp, SpotLtp, ATMStrike, t)
        #print(expiryDate)
        #print(CeLtp, SpotLtp, ATMStrike, fabs(max(SpotLtp - ATMStrike, 0.0)), t)
        try:
            AtmCeIv = implied_volatility(CeLtp, SpotLtp, ATMStrike, t, 0.1, 'c')
        except Exception:
            print(date, stock, ' Celtp < intrinsic | formula')
            AtmCeIv = 0
            
    if PeLtp is None:
        PeLtp = 0
        AtmPeIv = 0
        print('No PE ATm contract : ', date, stock)
    elif PeLtp < fabs(max(ATMStrike - SpotLtp, 0.0)):
        AtmPeIv = 0
        print(date, stock, ' Peltp < intrinsic')
    else:
        try:
            AtmPeIv = implied_volatility(PeLtp, SpotLtp, ATMStrike, t, 0.1, 'p')
        except Exception:
            print(date, stock, ' Peltp < intrinsic | formula')
            AtmPeIv = 0

    
    return SpotLtp, CeLtp, PeLtp, AtmCeIv, AtmPeIv



def getFutPrice(date, stock, path=path):
    
    try:
        allContracts = os.listdir(path+'/'+date+'/Fno/Stock/'+stock+'/Fut')
    except:
        print('No Fut Folder : ', date, stock)
        return 0
    if len(allContracts) == 0:
        print('There is no Fut contract')
        return 0
    
    
    pattern = re.compile(
        r'NSE_([A-Z0-9\-\&]+?)'   # Symbol (with dash/& allowed)
        r'(\d{2})'                # Year (2 digits)
        r'([A-Z]{3})'             # Month (3 letters)
        r'(?:'                    # Start non-capturing group
            r'(\d+(?:\.\d+)?)'    # Strike (integer or float) for options
            r'([CP]E)'            # Option type CE/PE
            r'|'                  # OR
            r'(FUT)'              # Futures
        r')'
        r'\.[a-zA-Z0-9]+$'        # File extension
    )

    expiry_month = []
    strikes = []
    for contract in allContracts:
        #print(contract)
        m = pattern.search(contract)
        symbol, year, mon, strike, opt_type, fut_type = m.groups()

        if mon not in expiry_month:
            expiry_month.append(mon)
        if strike not in strikes:
            strikes.append(strike)

    expiry = [ last_thursday(ex, 2025) for ex in expiry_month]
    #latestExpiry = sorted(expiry)[0]

    sorted_pairs = sorted(zip(expiry, expiry_month), key=lambda x: x[0], reverse=False)
    expiry, expiry_month = zip(*sorted_pairs)
    expiry = list(expiry)
    expiry_month = list(expiry_month)

    expiryDate = expiry[0]
    expiryMonth = expiry_month[0]


    ### then get the FUT Price
    FutLtp = None
    for contract in allContracts:
        m = pattern.search(contract)
        symbol, year, mon, strike, opt_type, fut_type = m.groups()

    #     if contract[-4:] == '.csv':
    #         Fut = contract.split('.csv')[0][-2:]
    #     else:
    #         Fut = contract.split('.parquet')[0][-2:]
        if mon == expiryMonth:
            if contract[-4:] == '.csv':
                Df = pd.read_csv(path+'/'+date+'/Fno/Stock/'+stock+'/Fut/'+contract)
            else:
                Df = pd.read_parquet(path+'/'+date+'/Fno/Stock/'+stock+'/Fut/'+contract)
            FutLtp = Df.set_index('Timestamp').sort_index().iloc[-1]['Close']

        else:
            pass
    if FutLtp is None:
        print('FutPrice : None', date, stock)


    return FutLtp


def convert_types(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()  # Convert arrays to lists
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

    
def replace_nan_inf(obj):
    if isinstance(obj, dict):
        return {k: replace_nan_inf(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan_inf(v) for v in obj]
    elif isinstance(obj, (int, float)):
        if math.isnan(obj) or math.isinf(obj):
            return 0
        return obj
    return obj

In [2]:
mainDic = defaultdict(dict)

for date in dates:
    print(date)
    stocks = os.listdir(path + '/' + date + '/Fno/Stock')
    for stock in stocks:
        SpotLtp, CeLtp, PeLtp, AtmCeIv, AtmPeIv = getAtmCePeIV(date, stock)
        FutLtp = getFutPrice(date, stock)

        mainDic[date][stock] = {
            'SpotLtp': round(SpotLtp, 2),
            'FutLtp': round(FutLtp, 2),
            'CeLtp': round(CeLtp, 2),
            'PeLtp': round(PeLtp, 2),
            'AtmCeIv': round(AtmCeIv, 2),
            'AtmPeIv': round(AtmPeIv, 2)
        }


# Convert defaultdict to normal dict for JSON serialization
mainDic = dict(mainDic)

with open("IvSpotFut.json", "w") as f:
    json.dump(mainDic, f, indent=4, default=convert_types)

2025-05-12
2025-05-13
No PE ATm contract :  2025-05-13 PAGEIND
No PE ATm contract :  2025-05-13 BOSCHLTD
2025-05-14
No PE ATm contract :  2025-05-14 PAGEIND
No PE ATm contract :  2025-05-14 CDSL
2025-05-15
No CE ATm contract :  2025-05-15 MANAPPURAM
No PE ATm contract :  2025-05-15 DEEPAKNTR
No PE ATm contract :  2025-05-15 CDSL
No PE ATm contract :  2025-05-15 BSE
2025-05-16
No PE ATm contract :  2025-05-16 CDSL
No CE ATm contract :  2025-05-16 MANAPPURAM
2025-05-19
No CE ATm contract :  2025-05-19 MANAPPURAM
2025-05-20
2025-05-20 MANAPPURAM  Peltp < intrinsic
2025-05-21
2025-05-21 ABFRL  Celtp < intrinsic
2025-05-21 MANAPPURAM  Peltp < intrinsic
No CE ATm contract :  2025-05-21 MRF
2025-05-22
2025-05-23
2025-05-23 ITC  Celtp < intrinsic | formula
2025-05-26
2025-05-26 MANAPPURAM  Celtp < intrinsic | formula
2025-05-27
2025-05-28
2025-05-28 SBILIFE  Celtp < intrinsic
2025-05-28 VBL  Celtp < intrinsic | formula
2025-05-29
2025-05-29 ABB  Celtp < intrinsic
2025-05-29 ABB  Peltp < intrin

/home/sirshendu/mambaforge/lib/python3.10/site-packages/py_lets_be_rational/lets_be_rational.py:612: RuntimeWarning: divide by zero encountered in double_scalars
  return _unchecked_normalised_implied_volatility_from_a_transformed_rational_guess_with_limited_iterations(


2025-05-29 ACC  Peltp < intrinsic
2025-05-29 APLAPOLLO  Celtp < intrinsic | formula


/home/sirshendu/mambaforge/lib/python3.10/site-packages/py_lets_be_rational/lets_be_rational.py:612: RuntimeWarning: divide by zero encountered in double_scalars
  return _unchecked_normalised_implied_volatility_from_a_transformed_rational_guess_with_limited_iterations(


2025-05-29 AUBANK  Celtp < intrinsic | formula
2025-05-29 AUBANK  Peltp < intrinsic


...

2025-07-31 VEDL  Celtp < intrinsic | formula
2025-07-31 VEDL  Peltp < intrinsic | formula
2025-07-31 VOLTAS  Celtp < intrinsic | formula


/home/sirshendu/mambaforge/lib/python3.10/site-packages/py_lets_be_rational/lets_be_rational.py:612: RuntimeWarning: divide by zero encountered in double_scalars
  return _unchecked_normalised_implied_volatility_from_a_transformed_rational_guess_with_limited_iterations(


2025-07-31 WIPRO  Celtp < intrinsic
2025-07-31 WIPRO  Peltp < intrinsic | formula


/home/sirshendu/mambaforge/lib/python3.10/site-packages/py_lets_be_rational/lets_be_rational.py:612: RuntimeWarning: divide by zero encountered in double_scalars
  return _unchecked_normalised_implied_volatility_from_a_transformed_rational_guess_with_limited_iterations(


2025-07-31 YESBANK  Peltp < intrinsic | formula
2025-07-31 ZYDUSLIFE  Celtp < intrinsic | formula
2025-07-31 ZYDUSLIFE  Peltp < intrinsic | formula
2025-08-01
No CE ATm contract :  2025-08-01 BAJFINANCE
No CE ATm contract :  2025-08-01 IRB
No CE ATm contract :  2025-08-01 NUVAMA
2025-08-04
No CE ATm contract :  2025-08-04 BAJAJ-AUTO
No CE ATm contract :  2025-08-04 BAJFINANCE
No PE ATm contract :  2025-08-04 JSL
No PE ATm contract :  2025-08-04 MANKIND
2025-08-05
No PE ATm contract :  2025-08-05 360ONE
No CE ATm contract :  2025-08-05 PNBHOUSING
2025-08-06
No CE ATm contract :  2025-08-06 BAJAJ-AUTO
No CE ATm contract :  2025-08-06 BAJFINANCE
No CE ATm contract :  2025-08-06 CONCOR
No PE ATm contract :  2025-08-06 DALBHARAT
No CE ATm contract :  2025-08-06 POLICYBZR
2025-08-07
No CE ATm contract :  2025-08-07 BAJFINANCE
No CE ATm contract :  2025-08-07 CONCOR
No CE ATm contract :  2025-08-07 PNBHOUSING
2025-08-08
No PE ATm contract :  2025-08-08 360ONE
No PE ATm contract :  2025-08-08 

In [3]:
with open("IvSpotFut.json", "r") as f:
    data = json.load(f)
    
data = replace_nan_inf(data)

with open("IvSpotFut.json", "w") as f:
    json.dump(data, f, indent=4, default=convert_types)

## Fut OI

In [6]:
FutOIpath = '/media/sirshendu/One Touch/1_SMData/2_CurrentData/3_IntradayOptChain/1_Stocks'

dates = [d for d in os.listdir(FutOIpath) if is_valid_date(d)]
dates = sorted(dates, key=lambda x: datetime.datetime.strptime(x, "%Y-%m-%d"))

def extract_datetime(filename):
    return datetime.datetime.strptime(filename.replace(".json", ""), "%Y-%m-%d_%H-%M-%S")

In [7]:
mainDicFut = defaultdict(dict)

for date in dates:
    print(date)
    files = os.listdir(FutOIpath+"/"+date)
    latest_file = max(files, key=extract_datetime)
    
    with open(FutOIpath+"/"+date+"/"+latest_file, 'r') as f:
        data = json.load(f)

    #stocks = os.listdir(path + '/' + date + '/Fno/Stock')
    for stock in data.keys():
        #SpotLtp, CeLtp, PeLtp, AtmCeIv, AtmPeIv = getAtmCePeIV(date, stock)
        FutOI = data[stock]['FutureData']['FutOi']

        mainDicFut[date][stock] = {
            'FutOI': FutOI
        }


# Convert defaultdict to normal dict for JSON serialization
mainDicFut = dict(mainDicFut)

with open("FutOI.json", "w") as f:
    json.dump(mainDicFut, f, indent=4, default=convert_types)

2025-07-14
2025-07-15
2025-07-16
2025-07-17
2025-07-18
2025-07-21
2025-07-22
2025-07-23
2025-07-24
2025-07-25
2025-07-28
2025-07-29
2025-07-30
2025-07-31
2025-08-01
2025-08-04
2025-08-05
2025-08-06
2025-08-07
2025-08-08
2025-08-11
2025-08-12
2025-08-13
2025-08-14


In [8]:
with open("FutOI.json", "r") as f:
    data = json.load(f)
    
data = replace_nan_inf(data)

with open("FutOI.json", "w") as f:
    json.dump(data, f, indent=4, default=convert_types)